# 02. Gemini 에스컬레이션 확인

**목적:** Jev 확신도가 낮은 항목을 LLM이 재판단하는 단계와, LLM 단독 베이스라인을 검증한다.

확인할 것:
1. 에스컬레이션 항목에서 Gemini 판단이 라벨과 맞는지, Jev와 얼마나 일치하는지
2. 근거 인용(`evidence`)이 실제 답안 원문에 있는지 (환각 검사)
3. 하이브리드(Jev + 에스컬레이션) vs Gemini 단독: 라벨 일치, 채점자 부담, 비용, 지연

설계:
- 질문은 01에서 검증한 한국어 V2 질문 (`data/backprop_v1.json`)
- Gemini에게 Jev 결과를 보여주지 않음 (독립 재판단, 불일치 감지용)
- 답안당 1회 호출로 질문 전체를 판단 → 그 결과가 곧 베이스라인이고, 에스컬레이션 결과는 그중 해당 항목만 골라냄
- Gemini의 자기 보고 확신도는 받지 않음 (보정 안 됨). 대신 근거 인용을 검증

- 모델: Jev `jev-1.13.0`, Gemini `gemini-3.6-flash` (Stable, 버전 고정)
- 데이터: 직접 작성한 합성 답안이라 무료 등급 사용 가능. **실제 학생 답안은 유료 등급에서만** (무료 등급은 입력을 제품 개선에 사용하고 사람이 검토할 수 있음)
- 날짜: 2026-11-03

In [ ]:
import json, time
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from google import genai
from pydantic import BaseModel

from app.judges.jev import make_client, call_jev, parse_answers, apply_gate, is_auto, T_NOUL, T_OTHER

load_dotenv(find_dotenv(usecwd=True))

GEMINI_MODEL = "gemini-3.6-flash"
GEMINI_PRICE = {"input": 0.75, "output": 3.75}  # USD / 1M tokens (output includes thinking), until 2026-12-31; doubles from 2027-01-01
JEV_PRICE_INPUT = 0.042  # USD / 1M tokens, output free

OUT_DIR = Path("outputs/02_gemini_escalation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIXTURE = json.loads(Path("../../data/backprop_v1.json").read_text())
PROBLEM = FIXTURE["problem"]
ANSWERS = FIXTURE["answers"]
QUESTIONS = FIXTURE["questions"]["ko"]
EXPECTED = FIXTURE["expected"]

jev = make_client()
gemini = genai.Client()  # GOOGLE_API_KEY, then GEMINI_API_KEY

def make_state(answer: str) -> dict:
    return {"question": PROBLEM, "answer": answer}

## Jev 1차 판단
01과 같은 질문·임계값(`T_NOUL`, `T_OTHER`)으로 실행. `escalate` = skip도 자동 처리도 아닌 항목.

에스컬레이션 표본이 적으므로(01 기준 2칸) 분석용으로 `conf < 0.9` 구간(`wide`)도 함께 표시.

In [ ]:
jev_rows = []
for name, text in ANSWERS.items():
    data, latency = call_jev(jev, make_state(text), QUESTIONS)
    (OUT_DIR / f"jev_{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2))
    for j in apply_gate(parse_answers(data)):
        jev_rows.append({
            "answer": name, **asdict(j),
            "auto": not j.skipped and is_auto(j),
            "jev_latency": latency, "jev_input_tokens": data["usage"]["input_tokens"],
        })

df_jev = pd.DataFrame(jev_rows).rename(columns={"pred": "jev_pred", "conf": "jev_conf", "raw": "jev_raw"})
df_jev["escalate"] = ~df_jev.skipped & ~df_jev.auto
df_jev["wide"] = ~df_jev.skipped & (df_jev.jev_conf < 0.9)

print(f"에스컬레이션 {df_jev.escalate.sum()}칸, conf < 0.9 {df_jev.wide.sum()}칸 / skip 제외 {(~df_jev.skipped).sum()}칸")
display(df_jev[df_jev.wide][["answer", "question_id", "jev_raw", "jev_pred", "jev_conf", "escalate"]])

## 응답 스키마와 프롬프트
- noul → `verdict`(true/false), score → `level`(0부터)
- `evidence`는 답안 문장을 그대로 복사. 없으면 빈 리스트
- 질문 문구는 Jev와 같은 `instructions`/`criteria`를 사용 (같은 기준으로 판단해야 비교 가능)
- temperature는 지정하지 않음 (모델 기본값). `store=False`로 요청·응답을 서버에 저장하지 않음

In [ ]:
class ItemJudgment(BaseModel):
    question_id: str
    verdict: bool | None = None  # noul
    level: int | None = None     # score
    evidence: list[str]
    reasoning: str

class GeminiResult(BaseModel):
    items: list[ItemJudgment]

SYSTEM = """당신은 대학 서술형 답안 채점 보조자입니다. 채점 기준 질문마다 답안을 판단하지만, 점수를 확정하지는 않습니다.

규칙:
- 각 질문은 서로 독립적으로 판단한다.
- noul 질문: verdict에 true 또는 false를 쓰고 level은 null로 둔다.
- score 질문: level에 해당하는 수준 번호(0부터)를 쓰고 verdict는 null로 둔다.
- evidence: 판단 근거가 된 문장을 answer에서 글자 그대로 복사한다. 요약하거나 고쳐 쓰지 않는다. 근거 문장이 없으면 빈 리스트로 둔다.
- reasoning: 판단 이유를 한국어 1~2문장으로 쓴다.
- 주어진 모든 질문에 대해 question_id별로 정확히 하나의 항목을 출력한다."""

def format_question(qid: str, q: dict) -> str:
    lines = [f"### {qid} ({q['type']})", q["instructions"]]
    criteria = q.get("criteria")
    if q["type"] == "noul" and criteria:
        lines += [f"- true: {criteria['true']}", f"- false: {criteria['false']}"]
    elif q["type"] == "score":
        lines += [f"- {i}: {desc}" for i, desc in enumerate(criteria)]
    return "\n".join(lines)

def build_prompt(answer: str, question_ids: list[str]) -> str:
    questions = "\n\n".join(format_question(qid, QUESTIONS[qid]) for qid in question_ids)
    return f"## question (문제)\n{PROBLEM}\n\n## answer (답안)\n{answer}\n\n## 판단할 질문\n{questions}"

def call_gemini(answer: str, question_ids: list[str]):
    t0 = time.perf_counter()
    interaction = gemini.interactions.create(
        model=GEMINI_MODEL,
        system_instruction=SYSTEM,
        input=build_prompt(answer, question_ids),
        response_format={"type": "text", "mime_type": "application/json", "schema": GeminiResult.model_json_schema()},
        store=False,
    )
    return interaction, time.perf_counter() - t0

def usage_of(interaction) -> dict:
    u = interaction.usage
    return {
        "input_tokens": getattr(u, "total_input_tokens", None) or 0,
        "output_tokens": getattr(u, "total_output_tokens", None) or 0,
        "thought_tokens": getattr(u, "total_thought_tokens", None) or 0,
    }

print(build_prompt(ANSWERS["partial"], list(QUESTIONS)))

## 단건 원본 확인
파서를 쓰기 전에 `output_text`와 `usage`를 눈으로 확인.

In [ ]:
interaction, latency = call_gemini(ANSWERS["partial"], list(QUESTIONS))
print(f"latency: {latency:.2f}s")
print(interaction.usage)
print(json.dumps(json.loads(interaction.output_text), ensure_ascii=False, indent=2))

## 파서
noul은 `verdict`, score는 `level`을 판단값으로 사용. 타입에 맞는 필드가 null이면 `gemini_pred`가 None으로 남음 (스키마 위반으로 집계).

In [ ]:
def parse_gemini(interaction) -> list[dict]:
    result = GeminiResult.model_validate_json(interaction.output_text)
    rows = []
    for item in result.items:
        q = QUESTIONS.get(item.question_id)
        pred = None if q is None else (item.verdict if q["type"] == "noul" else item.level)
        rows.append({"question_id": item.question_id, "gemini_pred": pred, "evidence": item.evidence, "reasoning": item.reasoning})
    return rows

pd.DataFrame(parse_gemini(interaction))

## 전체 실행 (답안 6개 × 질문 7개, 6회 호출)

In [ ]:
gem_rows, gem_calls = [], []
for name, text in ANSWERS.items():
    interaction, latency = call_gemini(text, list(QUESTIONS))
    (OUT_DIR / f"gemini_{name}.json").write_text(interaction.output_text)
    gem_calls.append({"answer": name, "gemini_latency": latency, **usage_of(interaction)})
    for row in parse_gemini(interaction):
        gem_rows.append({"answer": name, **row})

df_gem = pd.DataFrame(gem_rows)
gem_calls = pd.DataFrame(gem_calls)

ids = df_gem.groupby("answer").question_id.apply(set)
mismatch = {a: set(QUESTIONS) ^ s for a, s in ids.items() if s != set(QUESTIONS)}
print("question_id 누락/초과:", mismatch or "없음")
print("판단값 null(스키마 위반):", int(df_gem.gemini_pred.isna().sum()))
display(gem_calls)

## 근거 검증
인용문이 답안에 글자 그대로 있는지. 공백은 정규화하고 양 끝 구두점·따옴표는 제거한 뒤 부분 문자열로 비교.

In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.split()).strip(" .,\"'“”‘’")

def evidence_found(row) -> bool:
    text = " ".join(ANSWERS[row.answer].split())
    return all(normalize(q) in text for q in row.evidence)

df_gem["n_evidence"] = df_gem.evidence.str.len()
df_gem["evidence_ok"] = df_gem.apply(evidence_found, axis=1)

with_ev = df_gem[df_gem.n_evidence > 0]
print(f"근거가 있는 항목 {len(with_ev)}/{len(df_gem)}, 그중 원문 일치 {with_ev.evidence_ok.sum()}/{len(with_ev)}")
display(df_gem[~df_gem.evidence_ok][["answer", "question_id", "evidence"]])

## Jev vs Gemini vs 라벨

In [ ]:
exp = pd.DataFrame([{"answer": a, "question_id": q, "expected": v} for a, qs in EXPECTED.items() for q, v in qs.items()])
df = df_jev.merge(df_gem, on=["answer", "question_id"], how="left").merge(exp, on=["answer", "question_id"], how="left")
df["agree"] = df.jev_pred == df.gemini_pred
df["jev_correct"] = df.jev_pred == df.expected
df["gemini_correct"] = df.gemini_pred == df.expected
df.to_csv(OUT_DIR / "merged.csv", index=False)

active = df[~df.skipped]
print(f"Jev-Gemini 일치: {active.agree.sum()}/{len(active)}")
labeled = active[active.expected.notna()]
print(f"라벨 일치  Jev {labeled.jev_correct.sum()}/{len(labeled)}  Gemini {labeled.gemini_correct.sum()}/{len(labeled)}")

cols = ["answer", "question_id", "jev_pred", "jev_conf", "gemini_pred", "expected", "evidence", "reasoning"]
print("\n에스컬레이션 항목")
display(df[df.escalate][cols])
print("conf < 0.9 구간")
display(df[df.wide & ~df.escalate][cols])
print("Jev-Gemini 불일치 (전체)")
display(active[~active.agree][cols])

## 하이브리드 라우팅
CLAUDE.md 파이프라인 규칙 (코드로 분기):
- `auto`: Jev 확신도 높음 → Jev 판단을 추천
- `llm`: 에스컬레이션 + Jev와 Gemini 일치 → Gemini 근거와 함께 추천
- `human`: 에스컬레이션 + 불일치 → 채점자 직접 판단 (추천 없음)

In [ ]:
def route(r) -> str:
    if r.skipped:
        return "skip"
    if r.auto:
        return "auto"
    return "llm" if r.agree else "human"

df["route"] = df.apply(route, axis=1)
df["hybrid_pred"] = df.apply(lambda r: r.jev_pred if r.route in ("auto", "llm") else None, axis=1)
display(df.route.value_counts())
display(df[df.route.isin(["llm", "human"])][["answer", "question_id", "route", "jev_pred", "jev_conf", "gemini_pred", "expected", "reasoning"]])

## 하이브리드 vs Gemini 단독
- 라벨 기준은 skip이 아니고 라벨이 있는 칸
- 비용: Gemini는 답안당 1회 호출. 하이브리드는 에스컬레이션 항목이 있는 답안에서만 호출한다고 보고, 이번 호출(질문 전체 포함)을 상한으로 계산

In [ ]:
lab = df[~df.skipped & df.expected.notna()]
rec = lab[lab.route != "human"]

calls = gem_calls.set_index("answer")
call_cost = (calls.input_tokens * GEMINI_PRICE["input"] + (calls.output_tokens + calls.thought_tokens) * GEMINI_PRICE["output"]) / 1e6
escalated_answers = df[df.escalate].answer.unique()
jev_cost = df_jev.groupby("answer").jev_input_tokens.first() * JEV_PRICE_INPUT / 1e6

summary = pd.DataFrame({
    "hybrid": {
        "추천 칸": f"{len(rec)}/{len(lab)}",
        "채점자 직접 판단": int((lab.route == "human").sum()),
        "추천 중 라벨 일치": f"{(rec.hybrid_pred == rec.expected).sum()}/{len(rec)}",
        "비용 USD (6답안)": round(jev_cost.sum() + call_cost[call_cost.index.isin(escalated_answers)].sum(), 5),
        "지연 중앙값 s": round(df_jev.groupby("answer").jev_latency.first().median()
                         + (calls.gemini_latency.median() if len(escalated_answers) else 0), 2),
    },
    "gemini_only": {
        "추천 칸": f"{len(lab)}/{len(lab)}",
        "채점자 직접 판단": 0,
        "추천 중 라벨 일치": f"{lab.gemini_correct.sum()}/{len(lab)}",
        "비용 USD (6답안)": round(call_cost.sum(), 5),
        "지연 중앙값 s": round(calls.gemini_latency.median(), 2),
    },
})
summary

## 관찰

### 1. 에스컬레이션
- 에스컬레이션 칸 수, Gemini 판단과 라벨:
- Jev와 일치(`llm`) / 불일치(`human`):

### 2. 근거 인용
- 원문 일치율:
- 실패 유형 (요약해서 인용, 없는 문장 등):

### 3. 하이브리드 vs Gemini 단독
- 라벨 일치:
- 채점자 직접 판단 칸:
- 비용·지연:

### 4. Gemini 응답 품질
- 스키마 위반, question_id 누락:
- reasoning 품질 (한국어, 판단과의 일관성):

### 설계 반영 사항
- `GeminiJudge` 입력/출력:
- 근거 검증 규칙 (원문에 없는 인용 처리):
- 라우팅 규칙 확정: